In [1]:
import pandas as pd
import numpy as np


In [2]:
members=pd.read_csv("/Users/acyuthgopalakrishnan/Desktop/churn-intelligence/data/raw/members_v3.csv")
members.head()

,msno,city,bd,gender,registered_via,registration_init_time
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,0,NaN,11,20110911
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,0,NaN,7,20110914
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,0,NaN,11,20110915
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,0,NaN,11,20110915
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32,female,9,20110915


In [3]:
train=pd.read_csv("/Users/acyuthgopalakrishnan/Desktop/churn-intelligence/data/raw/train_v2.csv")
train.head()

,msno,is_churn
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1


In [4]:
transactions=pd.read_csv("/Users/acyuthgopalakrishnan/Desktop/churn-intelligence/data/raw/transactions_v2.csv")
transactions.head()


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,++6eU4LsQ3UQ20ILS7d99XK8WbiVgbyYL4FUgzZR134=,32,90,298,298,0,20170131,20170504,0
1,++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOeA=,41,30,149,149,1,20150809,20190412,0
2,+/GXNtXWQVfKrEDqYAzcSw2xSPYMKWNj22m+5XkVQZc=,36,30,180,180,1,20170303,20170422,0
3,+/w1UrZwyka4C9oNH3+Q8fUf3fD8R3EwWrx57ODIsqk=,36,30,180,180,1,20170329,20170331,1
4,+00PGzKTYqtnb65mPKPyeHXcZEwqiEzktpQksaaSC3c=,41,30,99,99,1,20170323,20170423,0


In [5]:
users=pd.read_csv("/Users/acyuthgopalakrishnan/Desktop/churn-intelligence/data/raw/user_logs_v2.csv")
users.head()



,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
0,u9E91QDTvHLq6NXjEaWv8u4QIqhrHk72kE+w31Gnhdg=,20170331,8,4,0,1,21,18,6309.273
1,nTeWW/eOZA/UHKdD5L7DEqKKFTjaAj3ALLPoAWsU8n0=,20170330,2,2,1,0,9,11,2390.699
2,2UqkWXwZbIjs03dHLU9KHJNNEvEkZVzm69f3jCS+uLI=,20170331,52,3,5,3,84,110,23203.337
3,ycwLc+m2O0a85jSLALtr941AaZt9ai8Qwlg9n0Nql5U=,20170331,176,4,2,2,19,191,7100.454
4,EGcbTofOSOkMmQyN1NMLxHEXJ1yV3t/JdhGwQ9wXjnI=,20170331,2,1,0,1,112,93,28401.558


In [12]:
import sqlite3
from pathlib import Path
pd.options.display.float_format = '{:.2f}'.format
RAW       = Path("../data/raw")
SYNTHETIC = Path("../data/synthetic")
DB_PATH   = Path("../data/churn.db")

print("Libraries loaded")

Libraries loaded


In [13]:
members      = pd.read_csv(RAW / "members_v3.csv")
train        = pd.read_csv(RAW / "train_v2.csv")
transactions = pd.read_csv(RAW / "transactions_v2.csv")
user_logs    = pd.read_csv(RAW / "user_logs_v2.csv", nrows=500000)

print(f"members:      {members.shape}")
print(f"train:        {train.shape}")
print(f"transactions: {transactions.shape}")
print(f"user_logs:    {user_logs.shape} (sample)")

members:      (6769473, 6)
train:        (970960, 2)
transactions: (1431009, 9)
user_logs:    (500000, 9) (sample)


In [14]:
#  Null Checks & Basic Anomalies
print("=== MEMBERS NULLS ===")
print(members.isnull().sum())

print("\n=== MEMBERS AGE (bd) DISTRIBUTION ===")
print(members['bd'].describe())

print("\n=== TRANSACTIONS NULLS ===")
print(transactions.isnull().sum())

print("\n=== USER LOGS NULLS ===")
print(user_logs.isnull().sum())

=== MEMBERS NULLS ===
msno                            0
city                            0
bd                              0
gender                    4429505
registered_via                  0
registration_init_time          0
dtype: int64

=== MEMBERS AGE (bd) DISTRIBUTION ===
count   6769473.00
mean          9.80
std          17.93
min       -7168.00
25%           0.00
50%           0.00
75%          21.00
max        2016.00
Name: bd, dtype: float64

=== TRANSACTIONS NULLS ===
msno                      0
payment_method_id         0
payment_plan_days         0
plan_list_price           0
actual_amount_paid        0
is_auto_renew             0
transaction_date          0
membership_expire_date    0
is_cancel                 0
dtype: int64

=== USER LOGS NULLS ===
msno          0
date          0
num_25        0
num_50        0
num_75        0
num_985       0
num_100       0
num_unq       0
total_secs    0
dtype: int64


In [15]:
# Cleaning the Members Table
print("Cleaning Members table...")

# Handle Gender: Filling nulls with 'unknown' so the model can use 'missing' as a pattern
members['gender'] = members['gender'].fillna('unknown')

# Handle Age (bd): Clamp to realistic human ages (10 to 90)
# Anything outside this gets mapped to NaN, then we fill NaNs with the median
members['bd'] = members['bd'].apply(lambda x: x if 10 <= x <= 90 else np.nan)
median_age = members['bd'].median()
members['bd'] = members['bd'].fillna(median_age)

# Convert YYYYMMDD integers to proper datetimes
members['registration_init_time'] = pd.to_datetime(members['registration_init_time'], format='%Y%m%d')

print("Cleaned Members shape:", members.shape)
print("\nNew Age (bd) Distribution:")
print(members['bd'].describe())
print("\nGender Distribution:")
print(members['gender'].value_counts())

Cleaning Members table...
Cleaned Members shape: (6769473, 6)

New Age (bd) Distribution:
count   6769473.00
mean         27.80
std           6.02
min          10.00
25%          27.00
50%          27.00
75%          27.00
max          90.00
Name: bd, dtype: float64

Gender Distribution:
gender
unknown    4429505
male       1195355
female     1144613
Name: count, dtype: int64


In [16]:
# Cell 5 - Pushing to Database
print("Connecting to SQLite database...")
conn = sqlite3.connect(DB_PATH)

print("Writing members...")
members.to_sql('kk_members', conn, if_exists='replace', index=False)

print("Writing train (labels)...")
train.to_sql('kk_train', conn, if_exists='replace', index=False)

print("Writing transactions...")
transactions.to_sql('kk_transactions', conn, if_exists='replace', index=False)

print("Writing user logs (sample)...")
user_logs.to_sql('kk_user_logs_sample', conn, if_exists='replace', index=False)

conn.close()
print("All tables successfully committed to churn.db!")

Connecting to SQLite database...
Writing members...
Writing train (labels)...
Writing transactions...
Writing user logs (sample)...
All tables successfully committed to churn.db!
